# Human score versus GPT score

In [ ]:
import pandas as pd
from scipy import stats
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt
import numpy as np
import os
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
from bert_score import score as bert_score
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_rel
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
# Load Toxic-BERT model and tokenizer
model_name = "unitary/toxic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
# Define the file path
file_path = 

# Read the CSV file
df_m = pd.read_csv(file_path)

# Display the first few rows of the DataFrame
df_m.head(10)

In [ ]:
# Define the file path
file_path = 

# Read the CSV file
df_a = pd.read_csv(file_path)

# Display the first few rows of the DataFrame
df_a.head(10)

# LLM Evaluator Based on Transcript

In [ ]:
# Define the file path
file_path = 

# Read the CSV file
llm_trans_note_eval = pd.read_csv(file_path)

# Display the first few rows
llm_trans_note_eval.head()

In [ ]:
score_columns = [col for col in llm_trans_note_eval.columns if 'score' in col.lower()]
llm_trans_note_score = llm_trans_note_eval[score_columns]
llm_trans_note_score.head(10)


# LLM Evaluator Based on Note

In [ ]:
# Define the file path
file_path = 
# Read the CSV file
llm_note_note_eval = pd.read_csv(file_path)

# Display the first few rows
llm_note_note_eval.head()

In [ ]:
llm_note_note_score = llm_note_note_eval[score_columns]
llm_note_note_score.head(10)

# Statistics

In [ ]:
def calculation(phy, llm):
    column_mapping = {
        'Fluency': 'Linguistic Quality_Fluency_Score',
        'Coherence': 'Linguistic Quality_Coherence_Score',
        'Clarity': 'Linguistic Quality_Clarity_Score',
        'Brevity': 'Linguistic Quality_Brevity_Score',
        'Structuring': 'Linguistic Quality_Structuring_Score',
        'Relevance': 'Content Integrity_Relevance_Score',
        'Completeness': 'Content Integrity_Completeness_Score',
        'Factuality': 'Content Integrity_Factuality_Score',
        'Comprehension': 'Content Integrity_Comprehension_Score',
        'Prudence': 'Trustworthiness_Prudence_Score',
        'Toxicity': 'Trustworthiness_Toxicity_Score'
    }
    
    result = []
    
    for phy_col, llm_col in column_mapping.items():
        # Calculate Spearman correlation and p-value
        correlation, p_value = stats.spearmanr(phy[phy_col], llm[llm_col])
        
        # Calculate Cohen's kappa
        kappa = cohen_kappa_score(phy[phy_col], llm[llm_col])
        
        # Calculate agreement rate
        agreement_rate = (phy[phy_col] == llm[llm_col]).mean() * 100  # Convert to percentage
        
        # Append results
        result.append({
            'Dimension': phy_col,
            'Spearman_Correlation': correlation,
            'P_Value': p_value,
            'Cohen_Kappa': kappa,
            'Agreement_Rate (%)': agreement_rate
        })
    
    return pd.DataFrame(result)

## Agreement between Mahmoud and LLM evaluator(trans)

In [ ]:
res_trans_human_m = calculation(df_m, llm_trans_note_eval)
res_trans_human_m.head(12)

In [ ]:
print(f"Average Spearman Correlation: {res_trans_human_m['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {res_trans_human_m['Agreement_Rate (%)'].mean()}")

## Agreement between Ankit and LLM evaluator(transcripts)


In [ ]:
res_trans_human_a = calculation(df_a, llm_trans_note_eval)
res_trans_human_a.head(12)

In [ ]:
print(f"Average Spearman Correlation: {res_trans_human_a['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {res_trans_human_a['Agreement_Rate (%)'].mean()}")

## Agreement between Mahmoud and LLM evaluator(note)

In [ ]:
res_note_human_m = calculation(df_m, llm_note_note_eval)
res_note_human_m.head(12)

In [ ]:
print(f"Average Spearman Correlation: {res_note_human_m['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {res_note_human_m['Agreement_Rate (%)'].mean()}")

## Agreement between Ankit and LLM evaluator(notes)

In [ ]:
res_note_human_a = calculation(df_a, llm_note_note_eval)
res_note_human_a.head(12)

In [ ]:
print(f"Average Spearman Correlation: {res_note_human_a['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {res_note_human_a['Agreement_Rate (%)'].mean()}")

## Agreement between LLM evalutor result based on note or transcript

In [ ]:
def calculation_llm(phy, llm):
    column_mapping = {
        'Linguistic Quality_Fluency_Score': 'Linguistic Quality_Fluency_Score',
        'Linguistic Quality_Coherence_Score': 'Linguistic Quality_Coherence_Score',
        'Linguistic Quality_Clarity_Score': 'Linguistic Quality_Clarity_Score',
        'Linguistic Quality_Brevity_Score': 'Linguistic Quality_Brevity_Score',
        'Linguistic Quality_Structuring_Score': 'Linguistic Quality_Structuring_Score',
        'Content Integrity_Relevance_Score': 'Content Integrity_Relevance_Score',
        'Content Integrity_Completeness_Score': 'Content Integrity_Completeness_Score',
        'Content Integrity_Factuality_Score': 'Content Integrity_Factuality_Score',
        'Content Integrity_Comprehension_Score': 'Content Integrity_Comprehension_Score',
        'Trustworthiness_Prudence_Score': 'Trustworthiness_Prudence_Score',
        'Trustworthiness_Toxicity_Score': 'Trustworthiness_Toxicity_Score'
    }
    
    result = []
    
    for phy_col, llm_col in column_mapping.items():
        # Calculate Spearman correlation and p-value
        correlation, p_value = stats.spearmanr(phy[phy_col], llm[llm_col])
        
        # Calculate Cohen's kappa
        kappa = cohen_kappa_score(phy[phy_col], llm[llm_col])
        
        # Calculate agreement rate
        agreement_rate = (phy[phy_col] == llm[llm_col]).mean() * 100  # Convert to percentage
        
        # Append results
        result.append({
            'Dimension': phy_col,
            'Spearman_Correlation': correlation,
            'P_Value': p_value,
            'Cohen_Kappa': kappa,
            'Agreement_Rate (%)': agreement_rate
        })
    
    return pd.DataFrame(result)

In [ ]:
res_note_trans = calculation_llm(llm_trans_note_eval, llm_note_note_eval)
res_note_trans.head(12)

In [ ]:
# Print the results
print(f"Average Spearman Correlation: {res_note_trans['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {res_note_trans['Agreement_Rate (%)'].mean()}")

## Agreement between human note

In [ ]:
def calculation_llm(phy, llm):
    column_mapping = {
        'Fluency': 'Fluency',
        'Coherence': 'Coherence',
        'Clarity': 'Clarity',
        'Brevity': 'Brevity',
        'Structuring': 'Structuring',
        'Relevance': 'Relevance',
        'Completeness': 'Completeness',
        'Factuality': 'Factuality',
        'Comprehension': 'Comprehension',
        'Prudence': 'Prudence',
        'Toxicity' : 'Toxicity'
    }
    
    result = []
    
    for phy_col, llm_col in column_mapping.items():
        # Calculate Spearman correlation and p-value
        correlation, p_value = stats.spearmanr(phy[phy_col], llm[llm_col])
        
        # Calculate Cohen's kappa
        kappa = cohen_kappa_score(phy[phy_col], llm[llm_col])
        
        # Calculate agreement rate
        agreement_rate = (phy[phy_col] == llm[llm_col]).mean() * 100  # Convert to percentage
        
        # Append results
        result.append({
            'Dimension': phy_col,
            'Spearman_Correlation': correlation,
            'P_Value': p_value,
            'Cohen_Kappa': kappa,
            'Agreement_Rate (%)': agreement_rate
        })
    
    return pd.DataFrame(result)

In [ ]:
human_statistics_m_a = calculation_llm(df_m, df_a)
human_statistics_m_a.head(12)

In [ ]:
# Print the results
print(f"Average Spearman Correlation: {human_statistics_m_a['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {human_statistics_m_a['Agreement_Rate (%)'].mean()}")

## Sensitivity analysis on 3 point scale

In [ ]:
# List of columns to be converted
columns_to_convert = ['Fluency', 'Coherence', 'Clarity', 'Brevity', 
                      'Structuring', 'Relevance', 'Completeness', 
                      'Factuality', 'Comprehension', 'Prudence', 'Toxicity']

# Function to map 5-point scale to 3-point scale
def map_scale(value):
    if value in [1]:
        return 1
    elif value in [2, 3]:
        return 2
    elif value in [4, 5]:
        return 3
    else:
        return value  # Keep as is if not in the range

# Create new DataFrames with the transformed columns
df_m_3 = df_m.copy()
df_a_3 = df_a.copy()

# Apply the mapping to the specified columns
df_m_3[columns_to_convert] = df_m_3[columns_to_convert].applymap(map_scale)
df_a_3[columns_to_convert] = df_a_3[columns_to_convert].applymap(map_scale)

human_statistics_m_a_3 = calculation_llm(df_m_3, df_a_3)
human_statistics_m_a_3.head(12)

In [ ]:
# Print the results
print(f"Average Spearman Correlation: {human_statistics_m_a_3['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {human_statistics_m_a_3['Agreement_Rate (%)'].mean()}")

## Agreement between avergae human score and the llm as an evaluator

In [ ]:
# Create a new DataFrame by averaging the relevant columns
columns_to_average = [
    'Fluency', 'Coherence', 'Clarity', 'Brevity',
    'Structuring', 'Relevance', 'Completeness', 'Factuality',
    'Comprehension', 'Prudence', 'Toxicity'
]

# Keep 'Document Number' as is
df_average = df_m[['Document Number']].copy()

# Calculate the average of the specified columns
for col in columns_to_average:
    df_average[col] = (df_m[col] + df_a[col]) / 2

# Display the resulting DataFrame
print(df_average)

In [ ]:
# Create a new DataFrame by averaging the relevant columns
columns_to_average = [
    'Fluency', 'Coherence', 'Clarity', 'Brevity',
    'Structuring', 'Relevance', 'Completeness', 'Factuality',
    'Comprehension', 'Prudence', 'Toxicity'
]

# Keep 'Document Number' as is
df_average_i = df_m[['Document Number']].copy()

# Calculate the average of the specified columns
for col in columns_to_average:
    df_average_i[col] = round((df_m[col] + df_a[col]) / 2)

# Display the resulting DataFrame
print(df_average_i)

### LLM based on transcript

In [ ]:
def calculation_average(phy, llm):
    column_mapping = {
        'Fluency': 'Linguistic Quality_Fluency_Score',
        'Coherence': 'Linguistic Quality_Coherence_Score',
        'Clarity': 'Linguistic Quality_Clarity_Score',
        'Brevity': 'Linguistic Quality_Brevity_Score',
        'Structuring': 'Linguistic Quality_Structuring_Score',
        'Relevance': 'Content Integrity_Relevance_Score',
        'Completeness': 'Content Integrity_Completeness_Score',
        'Factuality': 'Content Integrity_Factuality_Score',
        'Comprehension': 'Content Integrity_Comprehension_Score',
        'Prudence': 'Trustworthiness_Prudence_Score',
    }
    
    result = []
    
    for phy_col, llm_col in column_mapping.items():
        # Calculate Spearman correlation and p-value
        correlation, p_value = stats.spearmanr(phy[phy_col], llm[llm_col])
        
        
        # Calculate agreement rate
        agreement_rate = (phy[phy_col] == llm[llm_col]).mean() * 100  # Convert to percentage
        
        # Append results
        result.append({
            'Dimension': phy_col,
            'Spearman_Correlation': correlation,
            'P_Value': p_value,
            'Agreement_Rate (%)': agreement_rate
        })
    
    return pd.DataFrame(result)

In [ ]:
res_trans_human_average = calculation_average(df_average_i, llm_trans_note_eval)
res_trans_human_average.head(12)

In [ ]:
# Print the results
print(f"Average Spearman Correlation: {res_trans_human_average['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {res_trans_human_average['Agreement_Rate (%)'].mean()}")

In [ ]:
res_note_human_average = calculation_average(df_average_i, llm_note_note_eval)
res_note_human_average.head(12)

In [ ]:
# Print the results
print(f"Average Spearman Correlation: {res_note_human_average['Spearman_Correlation'].mean()}")
print(f"Average Agreement Rate (%): {res_note_human_average['Agreement_Rate (%)'].mean()}")

# Visualization 

In [ ]:
res_trans_human_m['source'] = 'Transcriptions'  
res_note_human_m['source'] = 'Notes'  

# Merge the dataframes
merged_df = pd.concat([res_trans_human_m[['Dimension', 'Spearman_Correlation', 'Agreement_Rate (%)', 'source']], 
                       res_note_human_m[['Dimension', 'Spearman_Correlation', 'Agreement_Rate (%)', 'source']]])

# Create the figure and axes
fig, axes = plt.subplots(2, 1, figsize=(12, 12))  # Two rows, one column

# First plot: Spearman Correlation
for source in merged_df['source'].unique():
    subset = merged_df[merged_df['source'] == source]
    axes[0].bar(subset['Dimension'], subset['Spearman_Correlation'], label=source, alpha=0.6)

axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Spearman Correlation')
axes[0].set_title('Spearman Correlation Comparison')
axes[0].set_xticks(range(len(merged_df['Dimension'].unique())))
axes[0].set_xticklabels(merged_df['Dimension'].unique(), rotation=90)  # Rotate dimension labels
axes[0].legend(title='Source', loc='upper right')

# Second plot: Agreement Rate (%)
for source in merged_df['source'].unique():
    subset = merged_df[merged_df['source'] == source]
    axes[1].bar(subset['Dimension'], subset['Agreement_Rate (%)'], label=source, alpha=0.6)

axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Agreement Rate (%)')
axes[1].set_title('Agreement Rate Comparison')
axes[1].set_xticks(range(len(merged_df['Dimension'].unique())))
axes[1].set_xticklabels(merged_df['Dimension'].unique(), rotation=90)  # Rotate dimension labels
axes[1].legend(title='Source', loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
res_trans_human_a['source'] = 'Transcriptions'  
res_note_human_a['source'] = 'Notes'  

# Merge the dataframes
merged_df = pd.concat([res_trans_human_a[['Dimension', 'Spearman_Correlation', 'Agreement_Rate (%)', 'source']], 
                       res_note_human_a[['Dimension', 'Spearman_Correlation', 'Agreement_Rate (%)', 'source']]])

# Create the figure and axes
fig, axes = plt.subplots(2, 1, figsize=(12, 12))  # Two rows, one column

# First plot: Spearman Correlation
for source in merged_df['source'].unique():
    subset = merged_df[merged_df['source'] == source]
    axes[0].bar(subset['Dimension'], subset['Spearman_Correlation'], label=source, alpha=0.6)

axes[0].set_xlabel('Dimension')
axes[0].set_ylabel('Spearman Correlation')
axes[0].set_title('Spearman Correlation Comparison')
axes[0].set_xticks(range(len(merged_df['Dimension'].unique())))
axes[0].set_xticklabels(merged_df['Dimension'].unique(), rotation=90)  # Rotate dimension labels
axes[0].legend(title='Source', loc='upper right')

# Second plot: Agreement Rate (%)
for source in merged_df['source'].unique():
    subset = merged_df[merged_df['source'] == source]
    axes[1].bar(subset['Dimension'], subset['Agreement_Rate (%)'], label=source, alpha=0.6)

axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Agreement Rate (%)')
axes[1].set_title('Agreement Rate Comparison')
axes[1].set_xticks(range(len(merged_df['Dimension'].unique())))
axes[1].set_xticklabels(merged_df['Dimension'].unique(), rotation=90)  # Rotate dimension labels
axes[1].legend(title='Source', loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Sort the data by Agreement_Rate (%) for better visualization
res_sorted = res_note_human_average.sort_values('Agreement_Rate (%)', ascending=False)

# Create the barplot
plt.figure(figsize=(10, 6))
plt.bar(res_sorted['Dimension'], res_sorted['Agreement_Rate (%)'], color='skyblue')
plt.xlabel('Criteria', fontsize=14)
plt.ylabel('Agreement Rate (%)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.tight_layout()

plt.savefig(save_path, dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
# Sort the data by Agreement_Rate (%) for better visualization
res_sorted = res_trans_human_a.sort_values('Agreement_Rate (%)', ascending=False)

# Create the barplot
plt.figure(figsize=(10, 6))
plt.bar(res_sorted['Dimension'], res_sorted['Agreement_Rate (%)'], color='skyblue')
plt.xlabel('Criteria', fontsize=14)
plt.ylabel('Agreement Rate (%)', fontsize=14)
plt.title('Agreement Rate Between Human and LLM', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Sort the data by Agreement_Rate (%) for better visualization
res_sorted = human_statistics_m_a.sort_values('Agreement_Rate (%)', ascending=False)

# Create the barplot
plt.figure(figsize=(10, 6))
plt.bar(res_sorted['Dimension'], res_sorted['Agreement_Rate (%)'], color='skyblue')
plt.xlabel('Criteria', fontsize=14)
plt.ylabel('Agreement Rate (%)', fontsize=14)
plt.title('Agreement Rate Between Human Evaluators', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
column_mapping = {
    'Fluency': 'Linguistic Quality_Fluency_Score',
    'Coherence': 'Linguistic Quality_Coherence_Score',
    'Clarity': 'Linguistic Quality_Clarity_Score',
    'Brevity': 'Linguistic Quality_Brevity_Score',
    'Structuring': 'Linguistic Quality_Structuring_Score',
    'Relevance': 'Content Integrity_Relevance_Score',
    'Completeness': 'Content Integrity_Completeness_Score',
    'Factuality': 'Content Integrity_Factuality_Score',
    'Comprehension': 'Content Integrity_Comprehension_Score',
    'Prudence': 'Trustworthiness_Prudence_Score',
    'Toxicity': 'Trustworthiness_Toxicity_Score'
}

# Select only relevant columns from df_m and llm_trans_note_eval
df_m_selected = df_m[list(column_mapping.keys())]
llm_selected = llm_trans_note_eval[list(column_mapping.values())]

llm_scores_avg = llm_selected.mean()
# Select only the relevant columns from df_m and llm_trans_note_eval
df_m_selected = df_m[list(column_mapping.keys())]
llm_selected = llm_trans_note_eval[list(column_mapping.values())]

# Calculate average scores
df_m_avg = df_m_selected.mean()
llm_scores_avg = llm_selected.mean()

# Align scores for radar chart
human_scores = df_m_avg
llm_scores = llm_scores_avg

# Radar chart setup
labels = list(column_mapping.keys())
num_vars = len(labels)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Complete the loop

# Radar chart data
human_scores = human_scores.tolist()
llm_scores = llm_scores.tolist()

# Complete the loop for radar chart
human_scores += human_scores[:1]
llm_scores += llm_scores[:1]

# Plot radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, human_scores, label='Human Score', linewidth=2)
ax.fill(angles, human_scores, alpha=0.25)
ax.plot(angles, llm_scores, label='LLM Score', linewidth=2)
ax.fill(angles, llm_scores, alpha=0.25)

# Add labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# Add legend
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# Add title
plt.title("Human vs LLM Scores", y=1.1)
plt.show()

In [ ]:
# Select only relevant columns from df_a
df_a_selected = df_a[list(column_mapping.keys())]

# Calculate average scores for df_a
df_a_avg = df_a_selected.mean()

# Align scores for radar chart
human_scores = df_m_avg.tolist() + [df_m_avg[0]]  # Add first score at the end to close the loop
llm_scores = llm_scores_avg.tolist() + [llm_scores_avg[0]]
df_a_scores = df_a_avg.tolist() + [df_a_avg[0]]

# Radar chart setup
labels = list(column_mapping.keys())
num_vars = len(labels)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Complete the loop

# Plot radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, human_scores, label='Human Score (df_m)', linewidth=2)
ax.fill(angles, human_scores, alpha=0.25)

ax.plot(angles, llm_scores, label='LLM Score', linewidth=2)
ax.fill(angles, llm_scores, alpha=0.25)

ax.plot(angles, df_a_scores, label='Human Score (df_a)', linewidth=2)
ax.fill(angles, df_a_scores, alpha=0.25)

# Add labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# Add legend
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# Add title
plt.title("Human (df_m & df_a) vs LLM Scores", y=1.1)
plt.show()

In [ ]:
# Select only relevant columns for df_average
df_avg_selected = df_average[list(column_mapping.keys())]

# Calculate average scores for df_average
df_avg_scores = df_avg_selected.mean()

# Align scores for radar chart
average_scores = df_avg_scores.tolist() + [df_avg_scores[0]]  # Add first score at the end to close the loop

# Radar chart setup
labels = list(column_mapping.keys())
num_vars = len(labels)

# Compute angle for each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # Complete the loop

# Plot radar chart
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, human_scores, label='Human Score (df_m)', linewidth=2)
ax.fill(angles, human_scores, alpha=0.25)

ax.plot(angles, llm_scores, label='LLM Score', linewidth=2)
ax.fill(angles, llm_scores, alpha=0.25)

ax.plot(angles, df_a_scores, label='Human Score (df_a)', linewidth=2)
ax.fill(angles, df_a_scores, alpha=0.25)

ax.plot(angles, average_scores, label='Average Scores (df_average)', linewidth=2, linestyle='--')
ax.fill(angles, average_scores, alpha=0.25)

# Add labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels)

# Add legend
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

# Add title
plt.title("Human (df_m & df_a) vs LLM Scores vs Average Scores", y=1.1)
plt.show()


# Emsemble score

## Calculate Automated Metric for GPT Note

In [ ]:
# File path
file_path = 
# Read the CSV file
metric_result_df = pd.read_csv(file_path)
metric_result_df.head(10)

In [ ]:
metric_prep = metric_result_df[['name', 'gpt_note', 'llama_note', 'reference_note']].rename(
    columns={
        'name': 'file_name',
        'gpt_note': 'gpt_note',
        'llama_note': 'llama_note',
        'reference_note': 'reference_text'
    }
)

In [ ]:
# Function to calculate toxicity
def calculate_toxicity(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    scores = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return scores[0][1].item()  # Toxicity score

# Function to calculate BLEU scores
def calculate_bleu(reference, candidate, n):
    smoothing = SmoothingFunction().method4
    weights = tuple([1.0 / n] * n) + tuple([0] * (4 - n))
    return sentence_bleu([reference.split()], candidate.split(), weights=weights, smoothing_function=smoothing)

# Function to calculate BLEU cumulative scores
def calculate_bleu_cumulative(reference, candidate):
    smoothing = SmoothingFunction().method4
    bleu1 = sentence_bleu([reference.split()], candidate.split(), weights=(1, 0, 0, 0), smoothing_function=smoothing)
    bleu2 = sentence_bleu([reference.split()], candidate.split(), weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
    bleu3 = sentence_bleu([reference.split()], candidate.split(), weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing)
    bleu4 = sentence_bleu([reference.split()], candidate.split(), weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
    return bleu1, bleu2, bleu3, bleu4

# Function to calculate ROUGE scores
def calculate_rouge(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rouge3', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return scores

# Main function
def calculate_text_metrics(df, reference_col, text_cols):
    results = df.copy()
    for col in text_cols:
        rouge1_list, rouge2_list, rouge3_list, rougeL_list = [], [], [], []
        bleu1_list, bleu2_list, bleu3_list, bleu4_list = [], [], [], []
        bert_precision_list, bert_recall_list, bert_f1_list = [], [], []
        meteor_list = []
        toxicity_list = []

        for ref, text in tqdm(zip(results[reference_col], results[col]), total=len(results), desc=f"Processing {col}"):
            # ROUGE
            rouge_scores = calculate_rouge(ref, text)
            rouge1_list.append(rouge_scores['rouge1'].fmeasure)
            rouge2_list.append(rouge_scores['rouge2'].fmeasure)
            rouge3_list.append(rouge_scores['rouge3'].fmeasure)
            rougeL_list.append(rouge_scores['rougeL'].fmeasure)

            # BLEU
            bleu1_list.append(calculate_bleu(ref, text, 1))
            bleu2_list.append(calculate_bleu(ref, text, 2))
            bleu3_list.append(calculate_bleu(ref, text, 3))
            bleu4_list.append(calculate_bleu(ref, text, 4))

            # BLEU Cumulative
            bleu_cum1, bleu_cum2, bleu_cum3, bleu_cum4 = calculate_bleu_cumulative(ref, text)

            # METEOR
            try:
                meteor_list.append(meteor_score([ref.split()], text.split()))
            except:
                meteor_list.append(0.0)  # Handle empty texts

            # BERTScore
            P, R, F1 = bert_score([text], [ref], lang="en", verbose=False)
            bert_precision_list.append(P.mean().item())
            bert_recall_list.append(R.mean().item())
            bert_f1_list.append(F1.mean().item())

            # Toxicity
            toxicity_list.append(calculate_toxicity(text))

        # Append results as new columns
        results[f'{col}_rouge1'] = rouge1_list
        results[f'{col}_rouge2'] = rouge2_list
        results[f'{col}_rouge3'] = rouge3_list
        results[f'{col}_rougeL'] = rougeL_list
        results[f'{col}_bleu1'] = bleu1_list
        results[f'{col}_bleu2'] = bleu2_list
        results[f'{col}_bleu3'] = bleu3_list
        results[f'{col}_bleu4'] = bleu4_list
        results[f'{col}_meteor'] = meteor_list
        results[f'{col}_bert_precision'] = bert_precision_list
        results[f'{col}_bert_recall'] = bert_recall_list
        results[f'{col}_bert_f1'] = bert_f1_list
        results[f'{col}_toxicity'] = toxicity_list

    return results


In [ ]:
file_path = 

# Re-import the DataFrame
metric_results_df = pd.read_csv(file_path)

## Join the Automated Metric Results with Human Evaluation Results

In [ ]:
merged_df = pd.merge(df_m, metric_results_df, left_on='Document Number', right_on='file_name', how='inner')
print(merged_df.columns)

## Predicted Dimentions based on Automated Metrics

In [ ]:
# Define human score dimensions and GPT metrics
human_scores = ['Fluency', 'Coherence', 'Clarity', 'Brevity',
                'Structuring', 'Relevance', 'Completeness', 'Factuality',
                'Comprehension', 'Prudence']
gpt_metrics = ['gpt_note_rouge1', 'gpt_note_rouge2', 'gpt_note_rouge3',
               'gpt_note_rougeL', 'gpt_note_bleu1', 'gpt_note_bleu2',
               'gpt_note_bleu3', 'gpt_note_bleu4', 'gpt_note_meteor',
               'gpt_note_bert_precision', 'gpt_note_bert_recall',
               'gpt_note_bert_f1']

# Prepare the dataset for regression
X = merged_df[gpt_metrics]

# Create a dictionary to store models and predictions
models = {}
predictions = {}

# Loop through each human score dimension
for score in human_scores:
    # Define the target variable
    y = merged_df[score]
    
    # Initialize and train the model
    model = LinearRegression()
    model.fit(X, y)
    
    # Predict on the entire dataset
    y_pred = model.predict(X)
    
    # Scale predictions to the range [1, 5]
    scaler = MinMaxScaler(feature_range=(0, 5))
    y_pred_scaled = scaler.fit_transform(y_pred.reshape(-1, 1)).flatten()
    
    # Store the model and scaled predictions
    models[score] = model
    predictions[score] = y_pred_scaled
    
    # Evaluate the model (optional, for reference only)
    mse = mean_squared_error(y, y_pred_scaled)
    print(f'MSE for {score}: {mse}')

# Append the predicted dimensions to the original dataframe
for score in human_scores:
    merged_df[f'Predicted_{score}'] = predictions[score]

In [ ]:
merged_df['Predicted_Toxicity'] = 5

## Prepare Data for Radar Plot

In [ ]:
metric_dict = {
    column.replace("Predicted_", ""): merged_df[column].mean()
    for column in merged_df.columns if column.startswith("Predicted_")
}

# Display the dictionary
print(metric_dict)

In [ ]:
columns_to_average = ['Fluency', 'Coherence', 'Clarity', 'Brevity', 
                      'Structuring', 'Relevance', 'Completeness', 'Factuality',
                      'Comprehension', 'Prudence', 'Toxicity']

# Calculate the mean for each column and store in a dictionary
human_dict = {col: df_m[col].mean() for col in columns_to_average}

# Print the dictionary
print(human_dict)

In [ ]:
# Define the mapping of keys to column names
column_mapping = {
    'Fluency': 'Linguistic Quality_Fluency_Score',
    'Coherence': 'Linguistic Quality_Coherence_Score',
    'Clarity': 'Linguistic Quality_Clarity_Score',
    'Brevity': 'Linguistic Quality_Brevity_Score',
    'Structuring': 'Linguistic Quality_Structuring_Score',
    'Relevance': 'Content Integrity_Relevance_Score',
    'Completeness': 'Content Integrity_Completeness_Score',
    'Factuality': 'Content Integrity_Factuality_Score',
    'Comprehension': 'Content Integrity_Comprehension_Score',
    'Prudence': 'Trustworthiness_Prudence_Score',
    'Toxicity': 'Trustworthiness_Toxicity_Score'
}

# Calculate the mean for each column and store in a dictionary with the specified keys
llm_dict = {key: llm_trans_note_eval[column].mean() for key, column in column_mapping.items()}

# Print the dictionary
print(llm_dict)

In [ ]:
# Prepare data for radar plot
categories = list(metric_dict.keys())
values_metric = list(metric_dict.values())
values_human = list(human_dict.values())
values_llm = list(llm_dict.values())

# Close the plot by repeating the first value
values_metric.append(values_metric[0])
values_human.append(values_human[0])
values_llm.append(values_llm[0])

# Convert categories into angles
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += [angles[0]]  # Closing the radar chart

# Plot
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})

# Add the filled areas
ax.fill(angles, values_metric, color='blue', alpha=0.25, label='Metric')
ax.fill(angles, values_human, color='green', alpha=0.25, label='Human')
ax.fill(angles, values_llm, color='red', alpha=0.25, label='LLM')

# Add the outlines
ax.plot(angles, values_metric, color='blue', linewidth=2)
ax.plot(angles, values_human, color='green', linewidth=2)
ax.plot(angles, values_llm, color='red', linewidth=2)

# Customize the plot
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=10, color='grey')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)

# Title and legend
plt.title('Radar Plot of Human vs Metric vs LLM', fontsize=16, pad=20)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1.1))
plt.show()
